In [12]:
# Cell 1 — Install PyG (rerun every session, takes ~2min)
!pip install -q torch_geometric torch_geometric_temporal

In [13]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/graphshield'
DATA_DIR    = f'{PROJECT_DIR}/data/raw/elliptic_bitcoin_dataset'
RESULTS_DIR = f'{PROJECT_DIR}/results'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Cell 3 — Standard imports
import json, time, sys
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Torch :', torch.__version__)

Device: cuda
Torch : 2.10.0+cu128


In [17]:
# Cell 4 — Import shared code from src/ (once you create it)
sys.path.append(f'{PROJECT_DIR}/src')
# from data.loaders import load_elliptic   # uncomment after you write it

## Load Elliptic and build the PyG Data object

Three CSVs:
- `elliptic_txs_features.csv` — col 0 = txId, col 1 = timestep, rest = features
- `elliptic_txs_edgelist.csv` — txId1, txId2
- `elliptic_txs_classes.csv` — txId, class ('1' illicit, '2' licit, 'unknown')

In [18]:
features_raw = pd.read_csv(f'{DATA_DIR}/elliptic_txs_features.csv', header=None)
edges_raw    = pd.read_csv(f'{DATA_DIR}/elliptic_txs_edgelist.csv')
classes_raw  = pd.read_csv(f'{DATA_DIR}/elliptic_txs_classes.csv')

print(f'features: {features_raw.shape}')
print(f'edges:    {edges_raw.shape}')
print(f'classes:  {classes_raw.shape}')
print()
print('class distribution:')
print(classes_raw['class'].value_counts())

features: (203769, 167)
edges:    (234355, 2)
classes:  (203769, 2)

class distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64


In [19]:
# Name columns: 0=txId, 1=timestep, rest = features
n_features = features_raw.shape[1] - 2
features_raw.columns = ['txId', 'timestep'] + [f'f{i}' for i in range(n_features)]
print(f'Number of node features: {n_features}')

# Map txId -> contiguous index 0..N-1
txid_to_idx = {txid: i for i, txid in enumerate(features_raw['txId'].values)}
N = len(txid_to_idx)
print(f'Num nodes: {N}')

# x and timesteps
timesteps = torch.tensor(features_raw['timestep'].values, dtype=torch.long)
x = torch.tensor(features_raw.iloc[:, 2:].values, dtype=torch.float)
print(f'x shape:   {tuple(x.shape)}')
print(f'timesteps: {int(timesteps.min())}..{int(timesteps.max())}')

Number of node features: 165
Num nodes: 203769
x shape:   (203769, 165)
timesteps: 1..49


In [20]:
# Drop edges referencing txIds not in features (defensive)
valid = edges_raw['txId1'].isin(txid_to_idx) & edges_raw['txId2'].isin(txid_to_idx)
edges_clean = edges_raw[valid]
print(f'Edges: {len(edges_raw)} raw -> {len(edges_clean)} after filtering')

src = edges_clean['txId1'].map(txid_to_idx).values
dst = edges_clean['txId2'].map(txid_to_idx).values
edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)
print(f'edge_index shape: {tuple(edge_index.shape)}')

Edges: 234355 raw -> 234355 after filtering
edge_index shape: (2, 234355)


In [21]:
# Labels: '1' illicit -> 1, '2' licit -> 0, 'unknown' -> -1
def map_class(c):
    s = str(c).strip()
    if s == '1': return 1
    if s == '2': return 0
    return -1

classes_raw['label'] = classes_raw['class'].apply(map_class)
label_lookup = dict(zip(classes_raw['txId'], classes_raw['label']))

y = torch.tensor(
    [label_lookup.get(txid, -1) for txid in features_raw['txId'].values],
    dtype=torch.long,
)
print(f'Labels: licit={int((y==0).sum())}, illicit={int((y==1).sum())}, unknown={int((y==-1).sum())}')

Labels: licit=42019, illicit=4545, unknown=157205


In [22]:
# Standard temporal split (Weber et al. 2019)
labeled    = (y != -1)
train_mask = labeled & (timesteps <= 34)
test_mask  = labeled & (timesteps >= 35)

print(f'Train: {int(train_mask.sum())} nodes  '
      f'(illicit={int((y[train_mask]==1).sum())}, licit={int((y[train_mask]==0).sum())})')
print(f'Test : {int(test_mask.sum())} nodes  '
      f'(illicit={int((y[test_mask]==1).sum())}, licit={int((y[test_mask]==0).sum())})')

data = Data(
    x=x, edge_index=edge_index, y=y,
    train_mask=train_mask, test_mask=test_mask,
    timesteps=timesteps,
).to(device)
print(data)

Train: 29894 nodes  (illicit=3462, licit=26432)
Test : 16670 nodes  (illicit=1083, licit=15587)
Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], test_mask=[203769], timesteps=[203769])


## Baseline 1 — Logistic Regression
Features only. Non-graph floor.

In [23]:
X_train = x[train_mask].numpy()
y_train = y[train_mask].numpy()
X_test  = x[test_mask].numpy()
y_test  = y[test_mask].numpy()

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

t0 = time.time()
clf = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1)
clf.fit(X_train_s, y_train)
print(f'Trained in {time.time()-t0:.1f}s')

probs = clf.predict_proba(X_test_s)[:, 1]
preds = (probs > 0.5).astype(int)

logreg_metrics = {
    'illicit_f1':        float(f1_score(y_test, preds, pos_label=1)),
    'illicit_precision': float(precision_score(y_test, preds, pos_label=1, zero_division=0)),
    'illicit_recall':    float(recall_score(y_test, preds, pos_label=1)),
    'roc_auc':           float(roc_auc_score(y_test, probs)),
    'pr_auc':            float(average_precision_score(y_test, probs)),
}
print(json.dumps(logreg_metrics, indent=2))

Trained in 3.1s
{
  "illicit_f1": 0.30299152135658297,
  "illicit_precision": 0.18324303405572756,
  "illicit_recall": 0.8744228993536473,
  "roc_auc": 0.8816708125203893,
  "pr_auc": 0.2898823332395072
}


## Shared GNN training loop

In [24]:
def train_gnn(model, data, epochs=200, lr=0.01, weight_decay=5e-4, label='gnn', verbose=True):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    n_illicit = int((data.y[data.train_mask] == 1).sum())
    n_licit   = int((data.y[data.train_mask] == 0).sum())
    w = torch.tensor([1.0, n_licit / max(n_illicit, 1)], device=device, dtype=torch.float)
    if verbose:
        print(f'Class weights: licit=1.0, illicit={w[1]:.2f}')

    best_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        out  = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=w)
        loss.backward()
        opt.step()

        if epoch % 20 == 0 or epoch == epochs:
            model.eval()
            with torch.no_grad():
                out   = model(data.x, data.edge_index)
                preds = out.argmax(dim=1).cpu().numpy()
            yt, tm = data.y.cpu().numpy(), data.test_mask.cpu().numpy()
            f1 = f1_score(yt[tm], preds[tm], pos_label=1)
            if f1 > best_f1:
                best_f1 = f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if verbose:
                print(f'Epoch {epoch:3d} | loss {loss.item():.4f} | test illicit-F1 {f1:.4f}')

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out   = model(data.x, data.edge_index)
        probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
        preds = out.argmax(dim=1).cpu().numpy()
    yt, tm = data.y.cpu().numpy(), data.test_mask.cpu().numpy()

    metrics = {
        'illicit_f1':        float(f1_score(yt[tm], preds[tm], pos_label=1)),
        'illicit_precision': float(precision_score(yt[tm], preds[tm], pos_label=1, zero_division=0)),
        'illicit_recall':    float(recall_score(yt[tm], preds[tm], pos_label=1)),
        'roc_auc':           float(roc_auc_score(yt[tm], probs[tm])),
        'pr_auc':            float(average_precision_score(yt[tm], probs[tm])),
    }
    torch.save(best_state, f'{RESULTS_DIR}/{label}.pt')
    if verbose:
        print(f'Best checkpoint saved to {RESULTS_DIR}/{label}.pt')
    return metrics

In [25]:
class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden=128, out_dim=2, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.relu(self.conv2(h, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv3(h, edge_index)

torch.manual_seed(42)
gcn = GCN(in_dim=data.x.shape[1]).to(device)
gcn_metrics = train_gnn(gcn, data, epochs=200, label='gcn')
print(json.dumps(gcn_metrics, indent=2))

Class weights: licit=1.0, illicit=7.63
Epoch  20 | loss 0.3842 | test illicit-F1 0.1892
Epoch  40 | loss 0.3120 | test illicit-F1 0.2349
Epoch  60 | loss 0.2676 | test illicit-F1 0.3393
Epoch  80 | loss 0.2346 | test illicit-F1 0.3719
Epoch 100 | loss 0.2066 | test illicit-F1 0.4335
Epoch 120 | loss 0.1941 | test illicit-F1 0.4212
Epoch 140 | loss 0.1779 | test illicit-F1 0.4134
Epoch 160 | loss 0.1668 | test illicit-F1 0.4294
Epoch 180 | loss 0.1610 | test illicit-F1 0.4315
Epoch 200 | loss 0.1569 | test illicit-F1 0.4323
Best checkpoint saved to /content/drive/MyDrive/graphshield/results/gcn.pt
{
  "illicit_f1": 0.4335195530726257,
  "illicit_precision": 0.36329588014981273,
  "illicit_recall": 0.5373961218836565,
  "roc_auc": 0.8159705382252335,
  "pr_auc": 0.429251997460269
}


In [26]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_dim, hidden=128, out_dim=2, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.relu(self.conv2(h, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv3(h, edge_index)

torch.manual_seed(42)
sage = GraphSAGE(in_dim=data.x.shape[1]).to(device)
sage_metrics = train_gnn(sage, data, epochs=200, label='graphsage')
print(json.dumps(sage_metrics, indent=2))

Class weights: licit=1.0, illicit=7.63
Epoch  20 | loss 0.2223 | test illicit-F1 0.3393
Epoch  40 | loss 0.1353 | test illicit-F1 0.4720
Epoch  60 | loss 0.0936 | test illicit-F1 0.5819
Epoch  80 | loss 0.0725 | test illicit-F1 0.6083
Epoch 100 | loss 0.0637 | test illicit-F1 0.6696
Epoch 120 | loss 0.0586 | test illicit-F1 0.6622
Epoch 140 | loss 0.0526 | test illicit-F1 0.6450
Epoch 160 | loss 0.0500 | test illicit-F1 0.6728
Epoch 180 | loss 0.0479 | test illicit-F1 0.6966
Epoch 200 | loss 0.0476 | test illicit-F1 0.6562
Best checkpoint saved to /content/drive/MyDrive/graphshield/results/graphsage.pt
{
  "illicit_f1": 0.6965620328849028,
  "illicit_precision": 0.7564935064935064,
  "illicit_recall": 0.6454293628808865,
  "roc_auc": 0.8999320585892036,
  "pr_auc": 0.685322150809598
}


In [27]:
all_metrics = {
    'logreg':    logreg_metrics,
    'gcn':       gcn_metrics,
    'graphsage': sage_metrics,
}

out_path = f'{RESULTS_DIR}/baselines.json'
with open(out_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f'Saved -> {out_path}\n')

summary = pd.DataFrame(all_metrics).T.round(4)
print(summary)

Saved -> /content/drive/MyDrive/graphshield/results/baselines.json

           illicit_f1  illicit_precision  illicit_recall  roc_auc  pr_auc
logreg         0.3030             0.1832          0.8744   0.8817  0.2899
gcn            0.4335             0.3633          0.5374   0.8160  0.4293
graphsage      0.6966             0.7565          0.6454   0.8999  0.6853


In [28]:
# Pure inference example — load saved checkpoint, predict on test nodes
sage_loaded = GraphSAGE(in_dim=data.x.shape[1]).to(device)
sage_loaded.load_state_dict(torch.load(f'{RESULTS_DIR}/graphsage.pt'))
sage_loaded.eval()

with torch.no_grad():
    logits = sage_loaded(data.x, data.edge_index)
    probs  = F.softmax(logits, dim=1)[:, 1]   # P(illicit)
    preds  = logits.argmax(dim=1)

# Show predictions for the first 10 test nodes
test_idx = test_mask.nonzero(as_tuple=True)[0][:10]
for i in test_idx:
    i = int(i)
    print(f'node {i:6d} | true={int(data.y[i])} | pred={int(preds[i])} | P(illicit)={float(probs[i]):.3f}')

node 136276 | true=0 | pred=0 | P(illicit)=0.000
node 136277 | true=0 | pred=0 | P(illicit)=0.000
node 136278 | true=0 | pred=0 | P(illicit)=0.000
node 136279 | true=1 | pred=1 | P(illicit)=0.682
node 136280 | true=1 | pred=1 | P(illicit)=1.000
node 136282 | true=0 | pred=0 | P(illicit)=0.001
node 136285 | true=0 | pred=0 | P(illicit)=0.000
node 136287 | true=0 | pred=0 | P(illicit)=0.000
node 136288 | true=0 | pred=0 | P(illicit)=0.000
node 136291 | true=0 | pred=0 | P(illicit)=0.000


# Phase 3a — EvolveGCN

Discrete-time temporal GNN. Builds 49 snapshots (one per timestep) and evolves
GCN weights across them via an RNN.

Reference: Pareja et al. 2020, "EvolveGCN: Evolving Graph Convolutional Networks for Dynamic Graphs"
Expected illicit-F1: ~0.55–0.65 (should beat static GCN/SAGE from Phase 2).

In [2]:
import torch
TORCH = torch.__version__.split('+')[0]      # e.g. "2.4.0"
CUDA = 'cu' + torch.version.cuda.replace('.', '')  # e.g. "cu121"

!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install torch_geometric torch_geometric_temporal

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 133.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 133.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 128.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 12.9 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/graphshield'
DATA_DIR    = f'{PROJECT_DIR}/data/raw/elliptic_bitcoin_dataset'
RESULTS_DIR = f'{PROJECT_DIR}/results'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

Mounted at /content/drive


In [4]:
import json, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric_temporal.nn.recurrent import EvolveGCNH

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


## Build per-timestep snapshots

For each timestep t we precompute the edges where both endpoints belong to t.
The full feature matrix x is reused across snapshots (it's static).

In [5]:
features_raw = pd.read_csv(f'{DATA_DIR}/elliptic_txs_features.csv', header=None)
edges_raw    = pd.read_csv(f'{DATA_DIR}/elliptic_txs_edgelist.csv')
classes_raw  = pd.read_csv(f'{DATA_DIR}/elliptic_txs_classes.csv')

n_features = features_raw.shape[1] - 2
features_raw.columns = ['txId', 'timestep'] + [f'f{i}' for i in range(n_features)]
txid_to_idx = {txid: i for i, txid in enumerate(features_raw['txId'].values)}
N = len(txid_to_idx)

timesteps = torch.tensor(features_raw['timestep'].values, dtype=torch.long)
x = torch.tensor(features_raw.iloc[:, 2:].values, dtype=torch.float).to(device)

valid = edges_raw['txId1'].isin(txid_to_idx) & edges_raw['txId2'].isin(txid_to_idx)
edges_clean = edges_raw[valid]
src = edges_clean['txId1'].map(txid_to_idx).values
dst = edges_clean['txId2'].map(txid_to_idx).values
edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long).to(device)

def map_class(c):
    s = str(c).strip()
    if s == '1': return 1
    if s == '2': return 0
    return -1

classes_raw['label'] = classes_raw['class'].apply(map_class)
label_lookup = dict(zip(classes_raw['txId'], classes_raw['label']))
y = torch.tensor(
    [label_lookup.get(txid, -1) for txid in features_raw['txId'].values],
    dtype=torch.long,
).to(device)

timesteps = timesteps.to(device)
print(f'N={N}, features={n_features}, edges={edge_index.shape[1]}')
print(f'timesteps {int(timesteps.min())}..{int(timesteps.max())}')

N=203769, features=165, edges=234355
timesteps 1..49


In [6]:
NUM_TIMESTEPS = int(timesteps.max())  # 49
snapshot_edges = {}

for t in range(1, NUM_TIMESTEPS + 1):
    node_mask = (timesteps == t)
    src_in = node_mask[edge_index[0]]
    dst_in = node_mask[edge_index[1]]
    edge_mask = src_in & dst_in
    snapshot_edges[t] = edge_index[:, edge_mask].contiguous()

# Sanity check
counts = [snapshot_edges[t].shape[1] for t in range(1, NUM_TIMESTEPS + 1)]
print(f'Edges/snapshot — min: {min(counts)}, max: {max(counts)}, mean: {np.mean(counts):.0f}')

Edges/snapshot — min: 1168, max: 9164, mean: 4783


## EvolveGCN model

`EvolveGCNH` keeps internal RNN state that evolves the GCN weight matrix
across forward calls. Output channels = input channels (it preserves dim),
so we add a final Linear head for classification.

In [7]:
class RecurrentGCN(torch.nn.Module):
    def __init__(self, num_nodes, node_features, num_classes=2):
        super().__init__()
        self.recurrent = EvolveGCNH(num_of_nodes=num_nodes, in_channels=node_features)
        self.linear = torch.nn.Linear(node_features, num_classes)

    def forward(self, x, edge_index):
        h = self.recurrent(x, edge_index)
        h = F.relu(h)
        return self.linear(h)

torch.manual_seed(42)
model = RecurrentGCN(num_nodes=N, node_features=n_features).to(device)
print(model)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

RecurrentGCN(
  (recurrent): EvolveGCNH(
    (pooling_layer): TopKPooling(165, ratio=0.0008097404413821534, multiplier=1.0)
    (recurrent_layer): GRU(165, 165)
    (conv_layer): GCNConv_Fixed_W(165, 165)
  )
  (linear): Linear(in_features=165, out_features=2, bias=True)
)
Params: 192,062


## Training

One epoch = full pass through train snapshots (timesteps 1–34) in order.
Loss is averaged across snapshots, backward'd once, optimizer steps.
This is BPTT through the weight-evolution RNN.

In [8]:
# Class weights from train timesteps
labeled       = (y != -1)
train_node_mask = labeled & (timesteps <= 34)
test_node_mask  = labeled & (timesteps >= 35)

n_illicit = int((y[train_node_mask] == 1).sum())
n_licit   = int((y[train_node_mask] == 0).sum())
class_w   = torch.tensor([1.0, n_licit / max(n_illicit, 1)], device=device, dtype=torch.float)
print(f'Train labeled: illicit={n_illicit}, licit={n_licit}, weight ratio={class_w[1]:.2f}')
print(f'Test labeled:  illicit={int((y[test_node_mask]==1).sum())}, '
      f'licit={int((y[test_node_mask]==0).sum())}')

Train labeled: illicit=3462, licit=26432, weight ratio=7.63
Test labeled:  illicit=1083, licit=15587


In [9]:
for m in model.modules():
    if isinstance(m, EvolveGCNH):
        print([a for a in dir(m) if 'weight' in a.lower() or 'init' in a.lower()])
        break

['__init__', '__init_subclass__', 'call_super_init', 'initial_weight', 'reinitialize_weight', 'weight']


In [10]:
EPOCHS = 100
LR     = 0.005

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)

def evaluate():
    model.eval()
    all_probs = torch.zeros(N, device=device)
    all_preds = torch.zeros(N, dtype=torch.long, device=device)
    with torch.no_grad():
        for t in range(1, NUM_TIMESTEPS + 1):
            out = model(x, snapshot_edges[t])
            probs = F.softmax(out, dim=1)[:, 1]
            preds = out.argmax(dim=1)
            mask_t = (timesteps == t)
            all_probs[mask_t] = probs[mask_t]
            all_preds[mask_t] = preds[mask_t]

    yt = y.cpu().numpy()
    tm = test_node_mask.cpu().numpy()
    pr = all_probs.cpu().numpy()
    pd_ = all_preds.cpu().numpy()
    return {
        'illicit_f1':        float(f1_score(yt[tm], pd_[tm], pos_label=1)),
        'illicit_precision': float(precision_score(yt[tm], pd_[tm], pos_label=1, zero_division=0)),
        'illicit_recall':    float(recall_score(yt[tm], pd_[tm], pos_label=1)),
        'roc_auc':           float(roc_auc_score(yt[tm], pr[tm])),
        'pr_auc':            float(average_precision_score(yt[tm], pr[tm])),
    }

def reset_recurrent_state():
    """EvolveGCNH carries evolved weights across forward calls.
    Reset them at the start of each epoch so autograd graphs don't span epochs."""
    for m in model.modules():
        if isinstance(m, EvolveGCNH):
            m.reinitialize_weight()

best_f1, best_state = -1.0, None
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    reset_recurrent_state()              # <-- KEY FIX
    optimizer.zero_grad()
    losses = []
    for t in range(1, 35):
        out = model(x, snapshot_edges[t])
        mask_t = (timesteps == t) & labeled
        if mask_t.sum() == 0:
            continue
        loss_t = F.cross_entropy(out[mask_t], y[mask_t], weight=class_w)
        losses.append(loss_t)
    total_loss = torch.stack(losses).mean()
    total_loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == EPOCHS:
        m = evaluate()
        if m['illicit_f1'] > best_f1:
            best_f1 = m['illicit_f1']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f'Epoch {epoch:3d} | loss {total_loss.item():.4f} | '
              f"test illicit-F1 {m['illicit_f1']:.4f} | "
              f"AUC {m['roc_auc']:.4f} | "
              f'best {best_f1:.4f} | {time.time()-t0:.0f}s')

Epoch  10 | loss 0.7651 | test illicit-F1 0.2010 | AUC 0.7007 | best 0.2010 | 4s
Epoch  20 | loss 0.5355 | test illicit-F1 0.2348 | AUC 0.7647 | best 0.2348 | 7s
Epoch  30 | loss 0.5100 | test illicit-F1 0.1636 | AUC 0.5948 | best 0.2348 | 10s
Epoch  40 | loss 0.7088 | test illicit-F1 0.1276 | AUC 0.4835 | best 0.2348 | 13s
Epoch  50 | loss 1.0230 | test illicit-F1 0.2778 | AUC 0.7771 | best 0.2778 | 15s
Epoch  60 | loss 0.8525 | test illicit-F1 0.2940 | AUC 0.7932 | best 0.2940 | 18s
Epoch  70 | loss 0.4702 | test illicit-F1 0.2166 | AUC 0.7483 | best 0.2940 | 21s
Epoch  80 | loss 0.3610 | test illicit-F1 0.2369 | AUC 0.7924 | best 0.2940 | 24s
Epoch  90 | loss 0.3209 | test illicit-F1 0.2502 | AUC 0.7999 | best 0.2940 | 27s
Epoch 100 | loss 0.3016 | test illicit-F1 0.2493 | AUC 0.8221 | best 0.2940 | 30s


In [11]:
# Restore best weights and compute final metrics
model.load_state_dict(best_state)
final_metrics = evaluate()
print(json.dumps(final_metrics, indent=2))

torch.save(best_state, f'{RESULTS_DIR}/evolve_gcn.pt')

# Update aggregate file
all_metrics_path = f'{RESULTS_DIR}/baselines.json'
if os.path.exists(all_metrics_path):
    all_metrics = json.load(open(all_metrics_path))
else:
    all_metrics = {}
all_metrics['evolve_gcn'] = final_metrics
with open(all_metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)

print(f'\nSaved checkpoint -> {RESULTS_DIR}/evolve_gcn.pt')
print(f'Updated metrics  -> {all_metrics_path}')
print()
print(pd.DataFrame(all_metrics).T.round(4))

{
  "illicit_f1": 0.3078683834048641,
  "illicit_precision": 0.2230514096185738,
  "illicit_recall": 0.4967682363804247,
  "roc_auc": 0.7979063808945127,
  "pr_auc": 0.1922471527557256
}

Saved checkpoint -> /content/drive/MyDrive/graphshield/results/evolve_gcn.pt
Updated metrics  -> /content/drive/MyDrive/graphshield/results/baselines.json

            illicit_f1  illicit_precision  illicit_recall  roc_auc  pr_auc
logreg          0.3052             0.1848          0.8772   0.8822  0.2918
gcn             0.4639             0.4381          0.4931   0.8169  0.4571
graphsage       0.6931             0.7706          0.6297   0.8980  0.6968
evolve_gcn      0.3079             0.2231          0.4968   0.7979  0.1922


# Phase 3b — TGN (Temporal Graph Network)

Continuous-time temporal GNN with per-node memory modules.
Reference: Rossi et al. 2020.

Adapting TGN to Elliptic requires faking continuous time from 49 discrete
timesteps — we treat each transaction edge as an event timestamped by its
destination node's timestep. Memory state at each node's timestep is used
for classification.

Expected illicit-F1: ~0.55–0.70 if it trains well. Memory can collapse;
if F1 stays at 0 after 5 epochs, lower learning rate and retry.